Hotel Reservations Dataset


https://www.kaggle.com/datasets/ahsan81/hotel-reservations-classification-dataset/data

# 수상작 리뷰

### 1. 코드 흐름 요약 (Workflow)
전형적인 고성능 파이프라인은 다음과 같은 5단계로 구성됩니다.
1.  **데이터 탐색 (EDA):** `lead_time`(예약 시점과 도착일 사이의 기간)과 취소율 사이의 강한 상관관계 확인.
2.  **전처리 (Preprocessing):** `Booking_ID` 등 불필요한 식별자 제거, 범주형 변수(식별값 위주)의 인코딩.
3.  **특성 공학 (Feature Engineering):** 총 숙박 일수 계산(`no_of_weekend_nights` + `no_of_week_nights`), 1인당 평균 가격 산출 등.
4.  **모델링 (Modeling):** **XGBoost, LightGBM, Random Forest**와 같은 트리 기반 앙상블 모델 활용.
5.  **최적화 및 평가:** GridSearchCV를 활용한 하이퍼파라미터 튜닝 및 `F1-score`, `ROC-AUC`를 통한 성능 검증.

---

### 2. 새롭게 알게 된 내용 (Insights)
* **Lead Time의 중요성:** 예약일이 실제 도착일보다 훨씬 앞선 경우(Long lead time) 취소 확률이 비약적으로 높아진다는 데이터적 근거를 발견할 수 있습니다.
* **Special Requests의 반전:** 특별 요청 사항(`no_of_special_requests`)이 많을수록 해당 고객의 숙박 의지가 강해 취소율이 낮아지는 경향이 있습니다.
* **Market Segment 영향:** 온라인 예약 채널을 통한 예약이 오프라인에 비해 훨씬 변동성이 큽니다.

---

### 3. 어려웠던 내용 (Challenges)
* **불균형 데이터 처리:** 취소되지 않은 예약이 훨씬 많은 불균형(Imbalanced) 상황에서 모델이 다수 클래스에 편향되지 않도록 `SMOTE`나 `Class Weight` 조절이 필요합니다.
* **범주형 변수의 처리:** `type_of_meal_plan`이나 `room_type_reserved` 같은 변수를 단순히 One-hot 인코딩할지, 순서 의미를 부여할지에 대한 결정이 모델 성능에 민감하게 작용합니다.

---

### 4. 배울 점 (Takeaways)
* **도메인 지식의 힘:** 호텔 산업에서 '리드 타임'이 수익 관리에 얼마나 중요한 요소인지 데이터로 증명하며 비즈니스 인사이트를 도출하는 과정을 배울 수 있습니다.
* **특성 중요도 (Feature Importance) 분석:** 단순 예측을 넘어, 어떤 요소가 취소를 유발하는지 **SHAP** 등의 라이브러리로 분석하여 "설명 가능한 AI(XAI)"를 구현하는 법을 익히기 좋습니다.

---

### 5. 주요 코드 (Key Code Snippets)

이 태스크에서 가장 핵심적인 부분은 **데이터 전처리**와 **모델 학습**입니다.

```python
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. 데이터 로드 및 간단한 전처리
df = pd.read_csv('Hotel Reservations.csv')
df.drop('Booking_ID', axis=1, inplace=True)

# 2. 범주형 변수 인코딩 (Label Encoding)
le = LabelEncoder()
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# 3. 특성 생성 (총 숙박 기간)
df['total_nights'] = df['no_of_weekend_nights'] + df['no_of_week_nights']

# 4. 모델링 (XGBoost)
X = df.drop('booking_status', axis=1)
y = df['booking_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6)
model.fit(X_train, y_train)

# 5. 특성 중요도 확인
import matplotlib.pyplot as plt
pd.Series(model.feature_importances_, index=X.columns).sort_values().plot(kind='barh')
plt.title('Key Features Influencing Cancellation')
plt.show()
```